<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.2/blob/main/Concept_Preserving_Preprocessing_for_Leakage_Safe_Language_Representations_2605.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================================
# Concept-preserving preprocessing from captions_long_preprocessed.csv only
#
# Input:
#   captions_long_preprocessed.csv
#
# Outputs:
#   concept_preprocessing_output/
#     numeric_matrix.csv
#     concept_captions_only.csv
#     WordToken_features_concept_only.csv
#     preprocess_diagnostics.csv
#   concept_preprocessing_output.zip
# ===============================================================

import os
import re
import glob
import zipfile
import numpy as np
import pandas as pd

# ===============================================================
# 0. Settings
# ===============================================================

OUTDIR = "./concept_preprocessing_output"
os.makedirs(OUTDIR, exist_ok=True)

NUMERIC_MATRIX_CSV = os.path.join(OUTDIR, "numeric_matrix.csv")
CONCEPT_CAPTION_CSV = os.path.join(OUTDIR, "concept_captions_only.csv")
CONCEPT_TOKEN_CSV = os.path.join(OUTDIR, "WordToken_features_concept_only.csv")
DIAGNOSTIC_CSV = os.path.join(OUTDIR, "preprocess_diagnostics.csv")
ZIP_PATH = "./concept_preprocessing_output.zip"

META_COLS = ["row_id", "Copolymer_Name", "variant"]

NUMERIC_COLS = [
    "comp_1", "comp_2", "comp_3",
    "headgroup_score_norm",
    "glycerol_score_norm",
    "alkenyl_score_norm",
    "alkyl_chain_score_norm",
    "motif_score_max",
    "motif_breadth_count_0p5",
    "motif_selectivity_index",
    "Peak1_T2_ms",
    "Peak2_T2_ms",
    "Peak3_T2_ms",
    "Peak4_T2_ms",
    "Width_log10T2",
    "Weighted_logmean_T2_ms",
    "N_detected_peaks",
    "Fit_R2",
    "Ridge_alpha",
]

# ===============================================================
# 1. Load captions_long_preprocessed.csv
# ===============================================================

def find_file(patterns):
    hits = []
    for p in patterns:
        hits.extend(glob.glob(p, recursive=True))
    hits = sorted(set(hits), key=lambda x: os.path.getmtime(x), reverse=True)
    return hits[0] if hits else None

CAPTION_CSV = find_file([
    "./**/captions_long_preprocessed.csv",
    "./**/captions_long_preprocessed*.csv",
    "/content/**/captions_long_preprocessed.csv",
    "/content/**/captions_long_preprocessed*.csv",
    "/mnt/data/**/captions_long_preprocessed.csv",
    "/mnt/data/**/captions_long_preprocessed*.csv",
])

if CAPTION_CSV is None:
    from google.colab import files
    print("Upload captions_long_preprocessed.csv")
    files.upload()

    CAPTION_CSV = find_file([
        "./**/captions_long_preprocessed.csv",
        "./**/captions_long_preprocessed*.csv",
        "/content/**/captions_long_preprocessed.csv",
        "/content/**/captions_long_preprocessed*.csv",
    ])

if CAPTION_CSV is None:
    raise FileNotFoundError("captions_long_preprocessed.csv not found.")

df = pd.read_csv(CAPTION_CSV, encoding="utf-8-sig")

for c in META_COLS:
    if c not in df.columns:
        raise KeyError(f"Missing required column: {c}")

print("Loaded:", CAPTION_CSV)
print("Shape:", df.shape)

# ===============================================================
# 2. Helper functions
# ===============================================================

def normalize_variant(v):
    if pd.isna(v):
        return ""
    return str(v).strip().lower().replace("cond_", "").replace("+", "")

def normalize_name(x):
    if pd.isna(x):
        return ""

    s = str(x).strip()
    s = re.sub(r"\(.*?\)", "", s)
    s = s.replace("-", "_").replace("/", "_")
    s = re.sub(r"\s+", "", s)

    s = s.replace("4VBA", "VBA")
    s = s.replace("MEDSH", "MEDSAH")
    s = s.replace("TECL2", "TECL")

    parts = s.split("_")
    if len(parts) >= 3:
        parts = parts[:3]

    repl = {
        "PSSA": "pSSA",
        "AAM": "AAm",
        "AMPS": "AMPS",
        "NIPAM": "NIPAM",
        "HMA": "HMA",
        "TFEMA": "TFEMA",
        "HA": "HA",
        "TECL": "TECL",
        "TRCL": "TRCL",
        "DICL": "DICL",
        "VBA": "VBA",
        "HEA": "HEA",
        "DMAA": "DMAA",
        "MEDSAH": "MEDSAH",
    }

    return "_".join([repl.get(p.upper(), p) for p in parts])

def parse_name_parts(name):
    name = normalize_name(name)
    parts = name.split("_")
    while len(parts) < 3:
        parts.append("Unknown")
    return parts[:3]

def safe_float(x):
    try:
        v = float(x)
        if np.isfinite(v):
            return v
    except Exception:
        pass
    return np.nan

def remove_numeric_expressions(text):
    if pd.isna(text):
        return ""

    s = str(text)

    # ratios such as (0.435:0.435:0.1)
    s = re.sub(r"\([0-9\.\:\,\s\-]+\)", " ", s)

    # key=value or key is numeric
    s = re.sub(
        r"\b[A-Za-z0-9_/\-\+\(\)]+(?:\s*=\s*|\s+is\s+|\s+are\s+)"
        r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?(?:\s*(?:ms|s|ppm|%|°C|C))?",
        " ",
        s
    )

    # numbers with units
    s = re.sub(
        r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s*(?:ms|s|ppm|%|°C|C)\b",
        " ",
        s
    )

    # standalone numbers
    s = re.sub(
        r"(?<![A-Za-z_])[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?(?![A-Za-z_])",
        " ",
        s
    )

    s = re.sub(r"\s+", " ", s).strip()
    return s

def monomer_concept(m):
    mapping = {
        "AMPS": "anionic sulfonate hydrophilic ionic hydration-rich monomer",
        "pSSA": "aromatic sulfonate hydrophilic ionic aromatic monomer",
        "MEDSAH": "zwitterionic sulfobetaine hydrophilic hydration-rich monomer",
        "VBA": "aromatic carboxylate hydrophilic hydrogen-bonding monomer",
        "HEA": "hydroxyl hydrophilic hydrogen-bonding monomer",
        "NIPAM": "amide-bearing thermoresponsive hydrophilic monomer",
        "DMAA": "amide-bearing hydrophilic polar monomer",
        "AAm": "amide-bearing hydrophilic polar monomer",
        "HMA": "alkyl hydrophobic flexible hydrocarbon monomer",
        "TFEMA": "fluorinated hydrophobic low-surface-energy monomer",
        "HA": "alkyl hydrophobic flexible hydrocarbon monomer",
        "TECL": "low-crosslink-density network-forming crosslinker",
        "TRCL": "intermediate-crosslink-density network-forming crosslinker",
        "DICL": "high-crosslink-density network-forming crosslinker",
    }

    return mapping.get(str(m), f"{m} polymer component")

def motif_level(v):
    v = safe_float(v)

    if not np.isfinite(v):
        return "unknown"

    if v >= 0.67:
        return "strong"

    if v >= 0.33:
        return "moderate"

    if v > 0:
        return "weak"

    return "minimal"

def selectivity_concept(v):
    v = safe_float(v)

    if not np.isfinite(v):
        return "interaction selectivity undefined"

    if v >= 0.35:
        return "highly motif-selective interaction"

    if v >= 0.18:
        return "moderately motif-selective interaction"

    return "weakly selective distributed interaction"

def breadth_concept(v):
    v = safe_float(v)

    if not np.isfinite(v):
        return "interaction breadth undefined"

    if v >= 3:
        return "broad multi-motif perturbation"

    if v >= 2:
        return "dual-motif perturbation"

    if v >= 1:
        return "localized single-motif perturbation"

    return "minimal motif perturbation"

def dynamics_concept(row):
    wl = safe_float(row.get("Weighted_logmean_T2_ms", np.nan))
    width = safe_float(row.get("Width_log10T2", np.nan))
    npeak = safe_float(row.get("N_detected_peaks", np.nan))

    peaks = [
        safe_float(row.get("Peak1_T2_ms", np.nan)),
        safe_float(row.get("Peak2_T2_ms", np.nan)),
        safe_float(row.get("Peak3_T2_ms", np.nan)),
        safe_float(row.get("Peak4_T2_ms", np.nan)),
    ]
    peaks = [p for p in peaks if np.isfinite(p) and p > 0]

    phrases = []

    if np.isfinite(wl):
        if wl >= 100:
            phrases.append("mobile segmental dynamics")
        elif wl >= 10:
            phrases.append("intermediate segmental mobility")
        else:
            phrases.append("constrained segmental dynamics")

    if np.isfinite(width):
        if width >= 1.2:
            phrases.append("heterogeneous broad relaxation distribution")
        elif width >= 0.6:
            phrases.append("moderately heterogeneous relaxation distribution")
        else:
            phrases.append("narrow relaxation distribution")

    if np.isfinite(npeak):
        if npeak >= 3:
            phrases.append("multi-component relaxation landscape")
        elif npeak >= 2:
            phrases.append("two-component relaxation landscape")
        else:
            phrases.append("single dominant relaxation component")

    if len(peaks) > 0:
        if max(peaks) >= 100:
            phrases.append("long T2 component present")
        if min(peaks) <= 5:
            phrases.append("short T2 restricted component present")

    if len(phrases) == 0:
        phrases.append("ILT-derived dynamics concept unavailable")

    return ", ".join(phrases)

def process_concept(row):
    raw = str(row.get("raw_text", "")) + " " + str(row.get("text_for_tokenization", ""))

    phrases = ["thermally initiated radical polymerization"]

    if "DMSO" in raw.upper():
        phrases.append("polar aprotic solvent DMSO")

    if "AIBN" in raw.upper():
        phrases.append("AIBN initiator")

    if "OVERNIGHT" in raw.upper():
        phrases.append("overnight curing")

    phrases.append("crosslinked polymer network preparation")

    return ", ".join(phrases)

def build_concept_caption(row):
    variant = normalize_variant(row["variant"])

    mono1, mono2, mono3 = parse_name_parts(row["Copolymer_Name"])

    if "mono_1" in row and pd.notna(row.get("mono_1", np.nan)):
        mono1 = str(row.get("mono_1"))

    if "mono_2" in row and pd.notna(row.get("mono_2", np.nan)):
        mono2 = str(row.get("mono_2"))

    if "mono_3" in row and pd.notna(row.get("mono_3", np.nan)):
        mono3 = str(row.get("mono_3"))

    sentences = []

    if "s" in variant:
        sentences.append(
            "[S] Structure: crosslinked copolymer architecture composed of "
            f"hydrophilic unit {mono1}, hydrophobic unit {mono2}, and crosslinker {mono3}; "
            "polymer network topology is retained as structural concept."
        )

    if "c" in variant:
        sentences.append(
            "[C] Chemistry: "
            f"{monomer_concept(mono1)}; "
            f"{monomer_concept(mono2)}; "
            f"{monomer_concept(mono3)}; "
            "chemical concepts include hydration, polarity, hydrophobicity, ionic character, aromaticity, fluorination, hydrogen bonding, and network formation."
        )

    if "m" in variant:
        motif_scores = {
            "headgroup": safe_float(row.get("headgroup_score_norm", np.nan)),
            "glycerol": safe_float(row.get("glycerol_score_norm", np.nan)),
            "alkenyl": safe_float(row.get("alkenyl_score_norm", np.nan)),
            "alkyl_chain": safe_float(row.get("alkyl_chain_score_norm", np.nan)),
        }

        finite_scores = {k: v for k, v in motif_scores.items() if np.isfinite(v)}

        if len(finite_scores) > 0:
            dominant = max(finite_scores, key=finite_scores.get)
        else:
            dominant = str(row.get("dominant_motif", "unknown"))

        sentences.append(
            "[M] Motif: lipid-associated motif information is retained as qualitative motif labels; "
            f"dominant lipid motif is {dominant}; "
            f"headgroup association is {motif_level(motif_scores['headgroup'])}; "
            f"glycerol association is {motif_level(motif_scores['glycerol'])}; "
            f"alkenyl association is {motif_level(motif_scores['alkenyl'])}; "
            f"alkyl-chain association is {motif_level(motif_scores['alkyl_chain'])}."
        )

    if "i" in variant:
        sentences.append(
            "[I] Interaction: "
            f"{selectivity_concept(row.get('motif_selectivity_index', np.nan))}; "
            f"{breadth_concept(row.get('motif_breadth_count_0p5', np.nan))}; "
            "polymer-lipid interaction is described by localized versus broad perturbation and motif-dependent interfacial response."
        )

    if "d" in variant:
        sentences.append(
            "[D] Dynamics: "
            f"{dynamics_concept(row)}; "
            "ILT-derived dynamic concepts include mobility, heterogeneity, relaxation landscape, restricted component, and long-T2 component."
        )

    if "p" in variant:
        sentences.append(
            "[P] Process: "
            f"{process_concept(row)}; "
            "processing concept links solvent, initiator, thermal polymerization, curing history, and network formation."
        )

    if len(sentences) == 0:
        sentences.append(
            "[N] Numeric-only: quantitative composition, motif, interaction, and ILT descriptors are stored only in numeric_matrix."
        )

    return remove_numeric_expressions(" ".join(sentences))

def simple_tokenize(text):
    s = str(text)
    s = s.replace("[", " ").replace("]", " ")
    s = s.replace("/", " ")
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"[^A-Za-z0-9_\-\+]+", " ", s)
    return [t.strip() for t in s.split() if t.strip()]

# ===============================================================
# 3. numeric_matrix.csv
# ===============================================================

for c in NUMERIC_COLS:
    if c not in df.columns:
        df[c] = np.nan

    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

extra_meta_cols = []

for c in ["mono_1", "mono_2", "mono_3", "dominant_motif"]:
    if c in df.columns:
        extra_meta_cols.append(c)

numeric_matrix = df[META_COLS + extra_meta_cols + NUMERIC_COLS].copy()
numeric_matrix.to_csv(NUMERIC_MATRIX_CSV, index=False, encoding="utf-8-sig")

# ===============================================================
# 4. concept_captions_only.csv
# ===============================================================

df["variant_norm"] = df["variant"].apply(normalize_variant)
df["Copolymer_Name_norm"] = df["Copolymer_Name"].apply(normalize_name)

if "raw_text" in df.columns:
    df["raw_text_no_numbers"] = df["raw_text"].apply(remove_numeric_expressions)
else:
    df["raw_text_no_numbers"] = ""

if "text_for_tokenization" in df.columns:
    df["original_text_no_numbers"] = df["text_for_tokenization"].apply(remove_numeric_expressions)
else:
    df["original_text_no_numbers"] = ""

df["concept_text_for_tokenization"] = df.apply(build_concept_caption, axis=1)

concept_cols = (
    META_COLS
    + ["Copolymer_Name_norm", "variant_norm", "concept_text_for_tokenization"]
    + extra_meta_cols
    + ["raw_text_no_numbers", "original_text_no_numbers"]
)

concept_caption_df = df[concept_cols].copy()
concept_caption_df.to_csv(CONCEPT_CAPTION_CSV, index=False, encoding="utf-8-sig")

# ===============================================================
# 5. WordToken_features_concept_only.csv
# ===============================================================

token_lists = []
all_tokens = []

for txt in concept_caption_df["concept_text_for_tokenization"]:
    toks = simple_tokenize(txt)
    token_lists.append(toks)
    all_tokens.extend(toks)

STOPWORDS = set([
    "the", "a", "an", "and", "or", "of", "to", "as", "is", "are", "by",
    "with", "in", "on", "for", "from", "using", "uses", "use",
    "contains", "contained", "composed", "retained", "represented",
    "described", "description", "information", "concept", "concepts",
    "conceptual", "labels", "material", "polymer", "copolymer",
])

vocab = []

for t in sorted(set(all_tokens)):
    if t.lower() in STOPWORDS:
        continue

    if len(t) <= 1 and t not in ["S", "C", "I", "M", "P", "D", "N"]:
        continue

    vocab.append(t)

FORCE_TOKENS = [
    "S", "C", "I", "M", "P", "D", "N",
    "Structure", "Chemistry", "Interaction", "Motif", "Process", "Dynamics",
    "hydration", "polarity", "hydrophobicity", "ionic", "aromaticity",
    "fluorination", "hydrogen", "bonding", "network", "formation",
    "headgroup", "glycerol", "alkenyl", "alkyl-chain",
    "motif-selective", "localized", "broad", "perturbation",
    "mobility", "heterogeneity", "relaxation", "restricted", "long-T2",
]

for t in FORCE_TOKENS:
    if t not in vocab:
        vocab.append(t)

vocab = sorted(set(vocab))

rows = []

for i, row in concept_caption_df.iterrows():
    toks = token_lists[i]
    counts = {f"TOK_{t}": 0 for t in vocab}

    for t in toks:
        if t in vocab:
            counts[f"TOK_{t}"] += 1

    out = {
        "row_id": row["row_id"],
        "Copolymer_Name": row["Copolymer_Name"],
        "variant": row["variant"],
    }
    out.update(counts)
    rows.append(out)

wordtoken_df = pd.DataFrame(rows)
wordtoken_df.to_csv(CONCEPT_TOKEN_CSV, index=False, encoding="utf-8-sig")

# ===============================================================
# 6. Diagnostics
# ===============================================================

diag_rows = []

for _, row in concept_caption_df.iterrows():
    txt = str(row["concept_text_for_tokenization"])
    toks = simple_tokenize(txt)

    diag_rows.append({
        "row_id": row["row_id"],
        "Copolymer_Name": row["Copolymer_Name"],
        "variant": row["variant"],
        "variant_norm": row["variant_norm"],
        "has_S": int("[S]" in txt),
        "has_C": int("[C]" in txt),
        "has_I": int("[I]" in txt),
        "has_M": int("[M]" in txt),
        "has_P": int("[P]" in txt),
        "has_D": int("[D]" in txt),
        "has_N": int("[N]" in txt),
        "n_tokens": len(toks),
        "n_numeric_like_patterns_remaining": len(
            re.findall(
                r"(?<![A-Za-z_])[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?(?![A-Za-z_])",
                txt
            )
        ),
        "concept_text_preview": txt[:300],
    })

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(DIAGNOSTIC_CSV, index=False, encoding="utf-8-sig")

# ===============================================================
# 7. ZIP
# ===============================================================

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in [
        NUMERIC_MATRIX_CSV,
        CONCEPT_CAPTION_CSV,
        CONCEPT_TOKEN_CSV,
        DIAGNOSTIC_CSV,
    ]:
        if os.path.exists(fp):
            zf.write(fp, arcname=os.path.basename(fp))

print("\nDone.")
print("Input:", CAPTION_CSV)
print("Output directory:", OUTDIR)
print("ZIP:", ZIP_PATH)

print("\nGenerated files:")
print("-", NUMERIC_MATRIX_CSV)
print("-", CONCEPT_CAPTION_CSV)
print("-", CONCEPT_TOKEN_CSV)
print("-", DIAGNOSTIC_CSV)

print("\nShapes:")
print("numeric_matrix:", numeric_matrix.shape)
print("concept_captions_only:", concept_caption_df.shape)
print("WordToken_features_concept_only:", wordtoken_df.shape)
print("diagnostics:", diag_df.shape)

print("\nVariant concept coverage:")
display_cols = [
    "has_S", "has_C", "has_I", "has_M",
    "has_P", "has_D", "has_N",
    "n_numeric_like_patterns_remaining"
]
print(diag_df.groupby("variant")[display_cols].mean().round(3))